### Sentiment analysis on Amazon Reviews


##File Handling in Python


In [4]:
reviews=[]
targets=[]
with open('/content/amazon_cells_labelled.txt','r') as file:
  for line in file:
    print(line)
    break

So there is no way for me to plug it in here in the US unless I go by a converter.	0



In [5]:


with open('/content/amazon_cells_labelled.txt','r') as file:
  for line in file:
  # line=line.split("\t")
  # line[1] = line[1].strip()
  # print(repr(line))
    review,target=line.strip().split("\t")
    reviews.append(review)
    targets.append(int(target))

In [ ]:
reviews[:2]

['So there is no way for me to plug it in here in the US unless I go by a converter.',
 'Good case, Excellent value.']

In [ ]:
targets[:2]

[0, 1]

In [ ]:
targets.count(0)

500

In [ ]:
targets.count(1)

500

In [ ]:
pip install nltk

In [ ]:
import nltk

nltk.download('stopwords')      # srop words in english
nltk.download('wordnet')        # vocabullary for lemmatization
nltk.download('punkt_tab')      # rules for tokenization

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re # regular expression

In [ ]:
stop_words=stopwords.words('english')
lemmatizer=WordNetLemmatizer()

In [ ]:
stop_words[:10]

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an']

In [ ]:
len(stop_words)

198

In [ ]:
lemmatizer.lemmatize("dancing",pos='v')#Parts of speech : n-noun,v-verb,a-adjective

'dance'

In [ ]:
cleaned_reviews=[]

def preprocess(text):
  # convert reviews into lowercase
  text=text.lower()

  # remove punctuations and numbers
  text=re.sub(r'[^a-z \s]',"",text) # replace everything except a-z and single space with empty string

  # tokkenize : split reviews into indvidual word
  tokens=word_tokenize(text)

  # remove stop words
  tokens=[word for word in tokens if word not in stop_words]

  # lemmatize: convert tokens to its verb form
  tokens=[lemmatizer.lemmatize(word,pos='v') for word in tokens]


  return " ".join(tokens)# put together the tokens back to sentiment format


sample="Amazing product . Delivered promptly . 10/10"
preprocess(sample)


'amaze product deliver promptly'

In [ ]:
for i in reviews:
  cleaned_reviews.append(preprocess(i))

In [ ]:
cleaned_reviews[:3]

['way plug us unless go converter',
 'good case excellent value',
 'great jawbone']

# Train test split

In [ ]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(cleaned_reviews,targets,test_size=0.2,random_state=12,stratify=targets)

In [ ]:
len(cleaned_reviews)==len(targets)

True

# Feature Extraction for understanding

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
vectorizer=TfidfVectorizer()

x_train_features=vectorizer.fit_transform(x_train)

In [ ]:
x_train_features.shape

(800, 1324)

In [ ]:
vectorizer.get_feature_names_out()

array(['abhor', 'ability', 'able', ..., 'yet', 'youd', 'za'], dtype=object)

# Model Training

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
multinomial_nb=MultinomialNB

In [ ]:
model=Pipeline([
    ('feature_extractor',TfidfVectorizer()),
    ('classifier',MultinomialNB())
])


In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid={'classifier__alpha':[0.2,0.5,1,1.2,2,3,5]} # Corrected parameter name
grid_search=GridSearchCV(model,param_grid,refit=True,cv=5)
grid_search

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                                       ('classifier', MultinomialNB())]),
             param_grid={'classifier__alpha': [0.2, 0.5, 1, 1.2, 2, 3, 5]})

In [ ]:
grid_search.fit(x_train,y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                                       ('classifier', MultinomialNB())]),
             param_grid={'classifier__alpha': [0.2, 0.5, 1, 1.2, 2, 3, 5]})

In [ ]:
best_model=grid_search.best_estimator_
best_model

Pipeline(steps=[('feature_extractor', TfidfVectorizer()),
                ('classifier', MultinomialNB(alpha=1))])

# Train performance

> Add blockquote



In [ ]:
y_train_pred=best_model.predict(x_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [ ]:
accuracy_train=accuracy_score(y_train,y_train_pred)
precision_train=precision_score(y_train,y_train_pred)
recall_train=recall_score(y_train,y_train_pred)


accuracy_train,precision_train,recall_train

(0.95875, 0.9317647058823529, 0.99)

In [ ]:
y_pred_test = best_model.predict(x_test)

In [ ]:
accuracy_test=accuracy_score(y_test,y_pred_test)
precision_test=precision_score(y_test,y_pred_test)
recall_test=recall_score(y_test,y_pred_test)


accuracy_test,precision_test,recall_test

(0.815, 0.7837837837837838, 0.87)